In [1]:
import os
import fitz  # PyMuPDF
from tqdm import tqdm
DATA_PATH = "./data/"

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

documents = []

for file in tqdm(os.listdir(DATA_PATH)):
    if file.endswith(".pdf"):
        full_path = os.path.join(DATA_PATH, file)
        text = extract_text_from_pdf(full_path)
        documents.append({
            "filename": file,
            "text": text
        })

print("Total documents loaded:", len(documents))

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:05<00:00,  1.69s/it]

Total documents loaded: 3


In [2]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
        
    return chunks

In [3]:
all_chunks = []

for doc in documents:
    chunks = chunk_text(doc["text"])
    for chunk in chunks:
        all_chunks.append({
            "filename": doc["filename"],
            "content": chunk
        })

print("Total chunks created:", len(all_chunks))

Total chunks created: 5831


## Connect to Ollama

In [4]:
import requests
import json
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "mistral"

# def generate_answer(context, query):
    
#     prompt = f"""
# You are a helpful assistant.

# Use the following context to answer the question.

# Context:
# {context}

# Question:
# {query}

# Answer:
# """
#     response = requests.post(
#         OLLAMA_URL,
#         json={
#             "model": MODEL_NAME,
#             "prompt": prompt,
#             "stream": False
#         }
#     )
    
#     return response.json()["response"]
def generate_answer(context, query):
    
    prompt = f"""
You are a helpful assistant.

Use ONLY the context below to answer.

Context:
{context}

Question:
{query}

Answer:
"""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False
        }
    )

    data = response.json()

    if "response" in data:
        return data["response"]
    else:
        print("Ollama Error:", data)
        return "Error from Ollama"

## STEP 1 — Load Cross-Encoder Model

In [5]:
from sentence_transformers import CrossEncoder,SentenceTransformer

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
texts = [chunk["content"] for chunk in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/183 [00:00<?, ?it/s]

Embedding shape: (5831, 384)


## STEP 2 —Modify Retrieval to Top-20

In [7]:
def retrieve_top_k(query, top_k=20):
    
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding)
    
    distances, indices = index.search(query_embedding, top_k)
    
    retrieved_chunks = []
    
    for idx in indices[0]:
        retrieved_chunks.append(all_chunks[idx])
    
    return retrieved_chunks

In [8]:
# The FAISS index is a binary file containing dense vector representations of all document chunks in the knowledge base. 
# It enables fast approximate nearest-neighbor search
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Total vectors in index:", index.ntotal)

Total vectors in index: 5831


## STEP 3 — Rerank Function

In [9]:
def rerank_chunks(query, retrieved_chunks, top_n=5):
    
    # Create query-chunk pairs
    pairs = [(query, chunk["content"]) for chunk in retrieved_chunks]
    
    # Get relevance scores
    scores = reranker.predict(pairs)
    
    # Attach scores to chunks
    scored_chunks = list(zip(retrieved_chunks, scores))
    
    # Sort by score descending
    scored_chunks = sorted(scored_chunks, key=lambda x: x[1], reverse=True)
    
    # Select top_n
    top_chunks = [chunk for chunk, score in scored_chunks[:top_n]]
    
    return top_chunks

## STEP 4 — Full RAG Pipeline With Reranking

In [10]:
def rag_with_reranking(query):
    
    # Step 1: Retrieve Top 20
    retrieved_chunks = retrieve_top_k(query, top_k=20)
    
    print("Retrieved Chunks:", len(retrieved_chunks))
    
    # Step 2: Rerank
    top_chunks = rerank_chunks(query, retrieved_chunks, top_n=5)
    
    print("After Reranking:", len(top_chunks))
    
    # Step 3: Build context
    context = "\n\n".join([chunk["content"] for chunk in top_chunks])
    
    # Step 4: Final Answer
    answer = generate_answer(context, query)
    
    return answer

In [11]:
query = "Explain how classification problems are solved in deep learning "

response = rag_with_reranking(query)

print(response)

Retrieved Chunks: 20
After Reranking: 5
 In deep learning, classification problems are solved by utilizing complex models consisting of many successive transformations of data, also known as deep models. These models are chained together top to bottom. The data is passed through these layers, each layer learning to extract a different and more abstract representation of the data.

For classification problems, these models are typically trained with objective functions that evaluate how well the model predicts the correct class labels for the given input data. During training, the parameters of the model are adjusted to minimize the objective function, thus improving the model's ability to classify the data accurately.

An important aspect of deep learning for classification is the use of numerical optimization algorithms, as most objective functions do not have analytical solutions. This numerical optimization helps adjust the model's parameters to find a solution that best fits the tr